# IEEE-CIS: Historical model comparison

These experiments use the original prepared feature export and random split. Outputs are cleared in the public repository. The TiDB release compares XGBoost and LightGBM on the same chronological validation window using average precision. See the [release report](../artifacts/models/ieee_cis_pre_auth_v1_0/validation.json).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, average_precision_score

from xgboost import XGBClassifier
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


In [ ]:
CLEANED_FEATURE_FILE = PROJECT_ROOT / "artifacts/dashboard/cleaned_ieee_features.csv"
if not CLEANED_FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Missing prepared IEEE-CIS feature file: {CLEANED_FEATURE_FILE}. "
        "Export the cleaned feature matrix before running this comparison notebook."
    )

df = pd.read_csv(CLEANED_FEATURE_FILE)

y = df["isFraud"]
X = df.drop("isFraud", axis=1)


In [ ]:
from sklearn.model_selection import train_test_split

# تقسيم الداتا مع ضمان توازن الفئات
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # دي أهم خطوة عشان الـ Recall
)

In [ ]:
print("NaN in Train:", X_train.isnull().sum().sum())
print("NaN in Validation:", X_val.isnull().sum().sum())

In [ ]:
from xgboost import XGBClassifier

# حساب الوزن: (عدد الحالات السليمة / عدد حالات الاحتيال)
ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1])

# تعريف الموديل
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=ratio, # ده السحر اللي هيرفع الـ Recall
    use_label_encoder=False,
    eval_metric='aucpr' # خليه يركز على الـ Area Under PR Curve أثناء التعلم
)

model.fit(X_train, y_train)

In [ ]:
import plotly.graph_objects as go
from sklearn.metrics import precision_recall_curve, auc

y_probs = model.predict_proba(X_val)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, y_probs)
pr_auc = auc(recall, precision)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=recall, 
    y=precision,
    mode='lines',
    name=f'XGBoost (AUC={pr_auc:.3f})',
    # بنعرض الـ Threshold عند كل نقطة عشان تقرر هتستخدم أنهي واحد
    text=[f"Threshold: {round(t, 3)}" for t in thresholds],
    hovertemplate="<b>Recall: %{x:.3f}</b><br>Precision: %{y:.3f}<br>%{text}<extra></extra>"
))

fig.update_layout(
    title=f"Precision-Recall Curve (Interactive) - PR-AUC: {pr_auc:.4f}",
    xaxis_title="Recall )",
    yaxis_title="Precision",
    hovermode="closest",
    template="plotly_dark"
)

fig.show()

In [ ]:
best_threshold = 0.2 
y_pred_final = (y_probs >= best_threshold).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_val, y_pred_final))

LGBMC

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score

# تعريف الموديل مع تفعيل خاصية التوازن
lgbm_model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    is_unbalance=True, # ده السحر بتاع التوازن في LightGBM
    extra_trees=True,   # بيقلل الـ Overfitting وبيحسن الـ Precision
    random_state=42
)

lgbm_model.fit(X_train, y_train)

# حساب الـ PR-AUC للمقارنة
y_probs_lgbm = lgbm_model.predict_proba(X_val)[:, 1]
lgbm_pr_auc = average_precision_score(y_val, y_probs_lgbm)
print(f"LightGBM PR-AUC: {lgbm_pr_auc:.4f}")

In [ ]:
from sklearn.metrics import recall_score

# 1. توقع الكلاسات بناءً على Threshold معين (مثلاً 0.4)
y_probs = lgbm_model.predict_proba(X_val)[:, 1]
y_pred = (y_probs >= 0.4).astype(int) 

# 2. حساب الـ Recall
rec = recall_score(y_val, y_pred)

print(f"Recall Score: {rec:.4f}")
print(f"Recall Percentage: {rec*100:.2f}%")

In [ ]:
from sklearn.metrics import classification_report

print("Classification Report for LightGBM:")
print(classification_report(y_val, y_pred))

In [ ]:
import plotly.graph_objects as go
from sklearn.metrics import precision_recall_curve, auc

# 1. الحصول على الاحتمالات (Probabilities)
# تأكد إنك سميت الموديل lgbm_model أو غير الاسم حسب الكود عندك
y_probs_lgbm = lgbm_model.predict_proba(X_val)[:, 1]

# 2. حساب قيم الكيرف
precision, recall, thresholds = precision_recall_curve(y_val, y_probs_lgbm)
pr_auc = auc(recall, precision)

# 3. رسم الكيرف التفاعلي باستخدام Plotly
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=recall, 
    y=precision,
    mode='lines',
    name=f'LightGBM (PR-AUC: {pr_auc:.4f})',
    # بنعرض الـ Threshold عند كل نقطة عشان تختار الصح
    text=[f"Threshold: {t:.4f}" for t in thresholds],
    hovertemplate="<b>Recall: %{x:.3f}</b><br>" +
                  "<b>Precision: %{y:.3f}</b><br>" +
                  "%{text}<extra></extra>"
))

fig.update_layout(
    title="Interactive Precision-Recall Curve (LightGBM)",
    xaxis_title="Recall  Fraud)",
    yaxis_title="Precision Fraud)",
    hovermode="closest",
    width=800,
    height=600,
    template="plotly_dark" # أو plotly_dark لو بتحب الخلفية الغامقة
)

fig.show()